In [ ]:
import os
os.chdir("/Users/apple/Downloads/mouse")

import pandas as pd, numpy as np, re
import matplotlib.pyplot as plt
from matplotlib.font_manager import FontManager

plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'PingFang SC', 'DejaVu Sans']
plt.rcParams['font.family'] = 'sans-serif'

# ---- eggNOG annotations: gene -> function ----
name_desc = {}
with open('eggnog_out.emapper.annotations') as f:
    for line in f:
        if line.startswith('##') or line.startswith('#query') or not line.strip(): continue
        c = line.rstrip().split('\t')
        if len(c) < 9: continue
        if c[8] and c[8] != '-' and c[7] and c[7] != '-':
            name_desc.setdefault(c[8], c[7])

# ---- WT markers, two calling methods ----
md = pd.read_csv('species_marker_genes_wt.tsv',    sep='\t').drop_duplicates(['species','gene'])
hc = pd.read_csv('species_marker_genes_wt_hc.tsv', sep='\t').drop_duplicates(['species','gene'])
md_idx = md.set_index(['species','gene'])['log2fc']
hc_idx = hc.set_index(['species','gene'])['log2fc']


In [ ]:
# ---- niche-module classification (first match wins) ----
def classify(gene, func):
    gf, fn = gene.lower(), (func or '').lower()
    if re.search(r'peptidase|dipeptidase|gct[ab]|fld|hgd|gcd|gdh|arg[e g]|ggt|agcs|yjff|agmatine|putrescine|aminotransferase|branched-chain|oligopeptide|pdu|cutc', gf+' '+fn): return 'AA fermentation/peptide'
    if re.search(r'glycosyl hydrolase|glucosidase|galactosidase|amylase|fuculose|glucuronidase|glycosid|sus[cd]|lacz|lace|tvai|xyl|mtld|ino1|glg|fuca|nag|agua|arabin|mannos|sugar|fruct|raffinose|arba|lspl', gf+' '+fn): return 'Glycan/sugar utilization'
    if re.search(r'abc transporter|permease|tonb|exb[bd]|substrate-binding|symport|antiporter|sulp|transport', gf+' '+fn): return 'Transport/uptake'
    if re.search(r'adhesion|adhesin|cell wall|outer membrane|penicillin-binding|mrca|roda|pic\b|neuc|capsule|polysaccharide biosynth|autotransporter|von willebrand|bata', gf+' '+fn): return 'Adhesion/envelope'
    if re.search(r'heat shock|hsp|sigma|sigv|ccpa|spo0a|regulator|degq|cfg\b|stress|two-component', gf+' '+fn): return 'Regulation/stress'
    if re.search(r'nqr|fumarate reductase|succinate dehydrogenase|frd|sdh|atp synthase|oxidoreductase|respirat|cytochrome|quinol', gf+' '+fn): return 'Respiration/energy'
    if re.search(r'biotin|biof|cob\w|b12|coenzyme|flavin|menaquinone|coae|methylmalonyl|mutb|thiamine|riboflavin|folate', gf+' '+fn): return 'Cofactors/vitamins'
    if re.search(r'sec[e dfg]\b|protein export|signal peptide|type [iv] secretion|conjugal', gf+' '+fn): return 'Secretion'
    return None

SPECIES_MODULES = {
    'Muribaculum gordoncarteri': ['Glycan/sugar utilization', 'Transport/uptake', 'Regulation/stress'],
    'Bacteroides acidifaciens':  ['Glycan/sugar utilization', 'Adhesion/envelope', 'Respiration/energy'],
    'Clostridium_Q sp910575795': ['Transport/uptake', 'AA fermentation/peptide', 'Regulation/stress'],
    'Akkermansia muciniphila_A': ['Glycan/sugar utilization', 'Cofactors/vitamins'],
    'Bacteroides sp002491635':   ['Respiration/energy', 'Secretion'],
}

# niche-module gene pool per species: {species: {gene: {'mod':..., 'fc_md':..., 'fc_hc':...}}}
pool = {}
for sp, mods in SPECIES_MODULES.items():
    pool[sp] = {}
    for df in (md, hc):
        for _, r in df[df['species'] == sp].iterrows():
            mod = classify(r['gene'], name_desc.get(r['gene'], ''))
            if mod in mods:
                e = pool[sp].setdefault(r['gene'], {'mod': mod, 'fc_md': None, 'fc_hc': None})
                if df is md: e['fc_md'] = r['log2fc']
                else:        e['fc_hc'] = r['log2fc']

def top_genes(sp, n, shared_only=False, mod=None):
    items = {g: v for g, v in pool[sp].items()
             if (mod is None or v['mod'] == mod)
             and (not shared_only or (v['fc_md'] is not None and v['fc_hc'] is not None))}
    return sorted(items, key=lambda g: -max(v for v in [items[g]['fc_md'] or -9, items[g]['fc_hc'] or -9]))[:n]

# ---- per-species gene selection (max 10) ----
selection = {}
# Muribaculum: 4 shared glycan genes + 6 MD-only sugar genes
sel = top_genes('Muribaculum gordoncarteri', 4, shared_only=True, mod='Glycan/sugar utilization')
sel += [g for g in ['carA','dxs','gmd','ligA','pyrB','ugd'] if g in pool['Muribaculum gordoncarteri']]
selection['Muribaculum gordoncarteri'] = sel
# Clostridium_Q: 5 shared AA-fermentation genes + 5 MD-only
sel = top_genes('Clostridium_Q sp910575795', 5, shared_only=True, mod='AA fermentation/peptide')
sel += [g for g in ['argE','dacA1','ilvC','carB','yhbU_1'] if g in pool['Clostridium_Q sp910575795']]
selection['Clostridium_Q sp910575795'] = sel
# other species: top 10 by max fc across niche modules (unchanged logic)
for sp in ['Bacteroides acidifaciens', 'Akkermansia muciniphila_A', 'Bacteroides sp002491635']:
    selection[sp] = top_genes(sp, 10)

for sp, genes in selection.items():
    print(f'{sp}: {len(genes)} genes -> {genes}')


In [ ]:
# ---- paired-column bubble plot: genes on y, species (MD|HC) on x ----
md_p = md.set_index(['species','gene'])
hc_p = hc.set_index(['species','gene'])
SPECIES = list(selection)

fig, ax = plt.subplots(figsize=(8, 0.26 * sum(len(g) for g in selection.values()) + 3.5))
cmap = plt.get_cmap('Reds'); VMAX = 4.0

y = 0; yticks, ylabels, ycolors, block_edge = [], [], [], []
module_spans = []
for si, sp in enumerate(SPECIES):
    genes = selection[sp]
    ordered = []
    for mod in SPECIES_MODULES[sp]:
        mod_genes = [g for g in genes if pool[sp][g]['mod'] == mod]
        shared = sorted([g for g in mod_genes if pool[sp][g]['fc_md'] is not None and pool[sp][g]['fc_hc'] is not None],
                        key=lambda g: -pool[sp][g]['fc_md'])
        mdonly = sorted([g for g in mod_genes if not (pool[sp][g]['fc_md'] is not None and pool[sp][g]['fc_hc'] is not None)],
                        key=lambda g: -(pool[sp][g]['fc_md'] or pool[sp][g]['fc_hc'] or 0))
        if shared + mdonly:
            module_spans.append((mod, y + len(ordered), y + len(ordered) + len(shared + mdonly) - 1))
            ordered += shared + mdonly
    for g in ordered:
        v = pool[sp][g]
        for x, fc in [(2*si, v['fc_md']), (2*si + 1, v['fc_hc'])]:
            if fc is None:
                ax.scatter(x, y, s=10, facecolors='none', edgecolors='#bbbbbb', linewidths=0.5, zorder=1)
            else:
                src = md_p if (sp, g) in md_p.index else hc_p
                ax.scatter(x, y, s=26 + 240 * src.loc[(sp, g), 'pct_sp'],
                           color=cmap(np.clip(fc / VMAX, -1, 1)),
                           edgecolors='black', linewidths=0.35, zorder=2)
        yticks.append(y)
        ylabels.append(g)
        ycolors.append('black')
        y += 1
    block_edge.append(y - 0.5)

for e in block_edge[:-1]:
    ax.axhline(e, color='#999999', lw=0.6, ls='--')
for si, sp in enumerate(SPECIES):
    ax.axvline(2*si - 0.5, color='#cccccc', lw=0.5)
    ax.text(2*si + 0.5, -1.2, sp, ha='center', va='top', fontsize=6.5, rotation=25)
for si, sp in enumerate(SPECIES):
    ax.text(2*si,     -2.5, 'mMD', ha='center', fontsize=8, fontweight='bold')
    ax.text(2*si + 1, -2.5, 'Inf', ha='center', fontsize=8, fontweight='bold')

ax.set_yticks(yticks)
ax.set_yticklabels(ylabels, fontsize=6.5)
for lab, col in zip(ax.get_yticklabels(), ycolors):
    lab.set_color(col)
import matplotlib.transforms as mtransforms
tr = mtransforms.blended_transform_factory(ax.transAxes, ax.transData)
for mod, y0, y1 in module_spans:
    ax.plot([-0.055, -0.055], [y0, y1], lw=0.7, color='#777777',
            transform=tr, clip_on=False)
    ax.text(-0.085, (y0 + y1) / 2, mod, transform=tr, va='center', ha='center',
            fontsize=6, style='italic', color='#444444', rotation=90, clip_on=False)
ax.set_xticks([]); ax.set_xlim(-0.9, 2*len(SPECIES) - 0.1); ax.set_ylim(y - 0.5, -3.2)
for pct, lab in [(0.0, '0%'), (0.25, '25%'), (0.5, '50%'), (1.0, '100%')]:
    ax.scatter([], [], s=26 + 240 * pct, facecolors='#dddddd', edgecolors='black',
               linewidths=0.4, label=f'pct of cells = {lab}')
ax.legend(title='bubble size', loc='lower left', fontsize=7, title_fontsize=7,
          frameon=False, bbox_to_anchor=(0.0, -0.06))
sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(0, VMAX))
cb = plt.colorbar(sm, ax=ax, fraction=0.03, pad=0.02, location='right')
cb.set_label('log2FC', fontsize=8); cb.ax.tick_params(labelsize=7)
ax.set_title('WT species functional-niche gene atlas: ModuleDrops (mMD) vs Inflection cutoff (Inf)',
             fontsize=9.5, pad=12)
plt.show()
